# AireIQ Unidad I integrada
**Dimensión:** humedad. **Responsable:** Eberth Emmanuel Rosales Castillo.

Ejecutar desde Kernel → Restart Kernel and Run All Cells. Este cuaderno une S2, S3 y S4 sobre el CSV original. Genera nuevas carpetas de resultados y conserva las anteriores. La fuente está montada en solo lectura.

## Arquitectura y alcance
Arquitectura Lambda: la ruta batch analiza el histórico y prepara modelos; la ruta streaming se implementará en U2. La fuente original es ESP32/BME680 → Supabase; esta evaluación usa su exportación CSV → PySpark → Parquet → MLlib. El laboratorio ejecuta Spark `local[2]`, no un clúster multinodo.

Pregunta descriptiva: ¿cómo varía la humedad por hora y día y qué porcentaje de lecturas válidas queda fuera de p5–p95?

Pregunta predictiva U1: ¿qué tan precisamente se estima la humedad con temperatura, presión y hora del mismo instante?

La automatización asiste la implementación; el estudiante debe revisar, comprender y defender las decisiones. El resultado del modelo se informará incluso si no supera una referencia sencilla.


# AireIQ · S2: entorno y lectura de humedad
**Responsable:** Eberth Emmanuel Rosales Castillo.

**Descriptiva:** ¿Cómo varía la humedad según la hora y el día y qué porcentaje de lecturas válidas queda fuera del rango histórico p5–p95?

**Predictiva U1:** ¿Qué tan precisamente se estima la humedad con temperatura, presión y hora?

**Predictiva U2:** ¿Cuál será la humedad dentro de un minuto y saldrá del rango histórico?

Este cuaderno comprueba el entorno y la fuente. S3 añadirá calidad completa y Parquet; S4, modelos. El rango histórico no es un umbral sanitario. Arquitectura Lambda: Supabase alimentará batch y streaming. El CSV original se monta como solo lectura.

Referencia de entorno: https://github.com/262bigdata/lambda26/tree/main/pyspark


In [1]:
import sys
from pathlib import Path
import pyspark
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder.appName("AireIQ-S2-Humedad")
         .master("local[2]")
         .config("spark.sql.session.timeZone", "America/Lima")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())
print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("Master:", spark.sparkContext.master)
print("Modo de laboratorio: dos hilos locales; no es un cluster multinodo.")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 03:08:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/20 03:08:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/20 03:08:40 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/11/20 03:08:40 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Python: 3.10.22 (main, Oct  6 2026, 02:05:33) [GCC 14.2.0]
PySpark: 4.2.0
Master: local[2]
Modo de laboratorio: dos hilos locales; no es un cluster multinodo.


In [2]:
path = Path("/opt/aireiq/data/mediciones_aire_rows.csv")
assert path.is_file(), "Falta el CSV montado por Compose"
schema = T.StructType([
    T.StructField("id", T.StringType()),
    T.StructField("created_at", T.TimestampType()),
    T.StructField("dispositivo", T.StringType()),
    *[T.StructField(c, T.DoubleType()) for c in [
        "temperatura_c", "humedad_pct", "presion_hpa", "gas_ohms", "altura_m",
        "iaq_estimado", "eco2_estimado_ppm", "voc_indice_estimado"]],
    T.StructField("calidad_aire", T.StringType())
])
df = (spark.read.option("header", True).option("mode", "FAILFAST")
      .option("enforceSchema", False).schema(schema).csv(str(path)))
df.printSchema()
df.select("created_at", "dispositivo", "humedad_pct", "temperatura_c", "presion_hpa").show(5, truncate=False)


root
 |-- id: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- dispositivo: string (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- humedad_pct: double (nullable = true)
 |-- presion_hpa: double (nullable = true)
 |-- gas_ohms: double (nullable = true)
 |-- altura_m: double (nullable = true)
 |-- iaq_estimado: double (nullable = true)
 |-- eco2_estimado_ppm: double (nullable = true)
 |-- voc_indice_estimado: double (nullable = true)
 |-- calidad_aire: string (nullable = true)

+--------------------------+------------+-----------+-------------+-----------+
|created_at                |dispositivo |humedad_pct|temperatura_c|presion_hpa|
+--------------------------+------------+-----------+-------------+-----------+
|2026-10-21 23:52:48.394234|ESP32_AireIQ|37.3878    |18.7746      |647.35     |
|2026-10-21 23:53:50.091276|ESP32_AireIQ|37.3368    |18.7791      |647.34     |
|2026-10-21 23:54:51.627012|ESP32_AireIQ|37.251     |18.8374      |647.33

## Transformación y evaluación perezosa
Definir columnas y filtros construye un plan. `explain()` permite inspeccionarlo; `count()` y `show()` solicitan resultados. El filtro siguiente aplica la regla inicial del brief, no sustituye la validación completa de S3. No se eliminan registros del CSV.

In [3]:
validas = df.filter((F.col("gas_ohms") > 0) & F.col("humedad_pct").between(0, 100))
por_hora = (validas.withColumn("hora_lima", F.hour("created_at"))
            .groupBy("hora_lima")
            .agg(F.count("*").alias("lecturas"), F.avg("humedad_pct").alias("humedad_promedio_pct")))
por_hora.explain(True)
total = df.count()
conservadas = validas.count()
print({"total": total, "conservadas_regla_inicial": conservadas, "excluidas": total-conservadas})
assert total > 0 and 0 < conservadas <= total
assert df.filter(F.col("created_at").isNull()).count() == 0, "Hay fechas no interpretadas"
por_hora.orderBy("hora_lima").show(24)


== Parsed Logical Plan ==
'Aggregate ['hora_lima], ['hora_lima, 'count(*) AS lecturas#35, 'avg('humedad_pct) AS humedad_promedio_pct#36]
+- Project [id#0, created_at#1, dispositivo#2, temperatura_c#3, humedad_pct#4, presion_hpa#5, gas_ohms#6, altura_m#7, iaq_estimado#8, eco2_estimado_ppm#9, voc_indice_estimado#10, calidad_aire#11, hour(created_at#1, Some(America/Lima)) AS hora_lima#34]
   +- Filter ((gas_ohms#6 > cast(0 as double)) AND ((humedad_pct#4 >= cast(0 as double)) AND (humedad_pct#4 <= cast(100 as double))))
      +- Relation [id#0,created_at#1,dispositivo#2,temperatura_c#3,humedad_pct#4,presion_hpa#5,gas_ohms#6,altura_m#7,iaq_estimado#8,eco2_estimado_ppm#9,voc_indice_estimado#10,calidad_aire#11] csv

== Analyzed Logical Plan ==
hora_lima: int, lecturas: bigint, humedad_promedio_pct: double
Aggregate [hora_lima#34], [hora_lima#34, count(1) AS lecturas#35L, avg(humedad_pct#4) AS humedad_promedio_pct#36]
+- Project [id#0, created_at#1, dispositivo#2, temperatura_c#3, humedad_pct

In [4]:
# Ejemplo RDD requerido en S2: conteo por dispositivo.
conteos_rdd = (validas.select("dispositivo").rdd
               .map(lambda row: (row.dispositivo, 1))
               .reduceByKey(lambda a, b: a + b).collect())
print(conteos_rdd)
assert sum(n for _, n in conteos_rdd) == conservadas


[Stage 13:>                                                         (0 + 1) / 1]

[('ESP32_AireIQ', 12853)]


## Evidencia
Ejecutar todas las celdas y guardar el cuaderno ejecutado. No se han precargado salidas.
El promedio mostrado pondera lecturas, no minutos; S3 tratará el cambio de frecuencia de captura.
Siguiente etapa: nulos, duplicados, rangos, cuarentena y Parquet particionado por fecha, con lectura de vuelta.


# AireIQ · S3: calidad y Parquet de humedad
**Eberth Emmanuel Rosales Castillo — Unidad I.**

Objetivo: transformar el CSV original en datos analíticos trazables. Cada ejecución genera una carpeta nueva; no sobrescribe la fuente ni otras evidencias.

Reglas: campos esenciales presentes y convertibles, números finitos, humedad entre 0 y 100 %, temperatura entre −40 y 85 °C y presión entre 300 y 1100 hPa. Estos son límites amplios de admisibilidad, no rangos de confort ni umbrales aprendidos. Se conserva la presión propia de gran altitud. Se aplica la exclusión `gas_ohms <= 0` del brief a todas las dimensiones como criterio conservador de fallo del sensor; no implica que toda humedad de 100 % sea inválida.

Duplicados: clave `(dispositivo, created_at)`; conservar determinísticamente la fila admisible de menor ID numérico, con desempate por contenido. Las demás se registran en cuarentena. No se imputan mediciones ausentes.

Gold promedia por minuto para reducir el efecto de las dos frecuencias de captura. No rellena minutos faltantes y marca tramos separados por más de 10 minutos. Los futuros pares de pronóstico deberán además estar exactamente a un minuto.

Referencia: https://262bigdata.github.io/lambda26/proyecto-sello/u1-producto/


In [5]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math
from pyspark.sql import SparkSession, functions as F, types as T, Window

spark = (SparkSession.builder.appName("AireIQ-S3-Humedad").master("local[2]")
    .config("spark.sql.session.timeZone", "America/Lima")
    .config("spark.sql.shuffle.partitions", "4").getOrCreate())
source = Path("/opt/aireiq/data/mediciones_aire_rows.csv")
run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = Path("/opt/aireiq/outputs/s03") / run
out.mkdir(parents=True, exist_ok=False)
names = ["id", "created_at", "dispositivo", "temperatura_c", "humedad_pct", "presion_hpa", "gas_ohms", "altura_m", "iaq_estimado", "eco2_estimado_ppm", "voc_indice_estimado", "calidad_aire"]
# Bronze conserva texto para poder auditar valores que no se convierten.
schema = T.StructType([T.StructField(c, T.StringType(), True) for c in names])
raw = (spark.read.option("header", True).option("mode", "FAILFAST")
       .option("enforceSchema", False).schema(schema).csv(str(source))).cache()
total = raw.count()
assert total > 0
raw.write.mode("errorifexists").parquet(str(out / "bronze"))
print("Lecturas de entrada:", total)


26/11/20 03:08:52 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

Lecturas de entrada: 15690


In [6]:
numeric = names[3:11]
def validar(frame):
    d = frame.withColumn("raw_json", F.to_json(F.struct(*[F.col(c) for c in names])))
    for c in names:
        d = d.withColumn(c, F.when(F.length(F.trim(F.col(c))) == 0, F.lit(None)).otherwise(F.trim(F.col(c))))
    for c in numeric:
        d = d.withColumn(c, F.expr(f"try_cast(`{c}` as double)"))
    d = d.withColumn("created_at", F.expr("try_cast(created_at as timestamp)"))
    required = ["id", "created_at", "dispositivo", "temperatura_c", "humedad_pct", "presion_hpa", "gas_ohms"]
    reasons = [F.when(F.col(c).isNull(), F.lit("ausente_o_tipo_invalido:" + c)) for c in required]
    for c in numeric:
        reasons.append(F.when(F.isnan(c) | (F.abs(F.col(c)) == float("inf")), F.lit("no_finito:" + c)))
    for c, lo, hi in [("humedad_pct", 0, 100), ("temperatura_c", -40, 85), ("presion_hpa", 300, 1100)]:
        reasons.append(F.when(~F.col(c).between(lo, hi), F.lit("fuera_rango:" + c)))
    reasons.append(F.when(F.col("gas_ohms") <= 0, F.lit("fallo_sensor_gas")))
    return d.withColumn("motivos", F.filter(F.array(*reasons), lambda x: x.isNotNull()))

checked = validar(raw).cache()
invalid = checked.filter(F.size("motivos") > 0)
eligible = checked.filter(F.size("motivos") == 0)
order = Window.partitionBy("dispositivo", "created_at").orderBy(
    F.expr("try_cast(id as bigint)").asc_nulls_last(), F.col("id"), F.col("raw_json"))
ranked = eligible.withColumn("orden", F.row_number().over(order))
duplicates = ranked.filter("orden > 1").drop("orden").withColumn("motivos", F.array(F.lit("duplicado_dispositivo_fecha")))
quarantine = invalid.unionByName(duplicates).cache()
silver = (ranked.filter("orden = 1").drop("orden", "motivos", "raw_json")
    .withColumn("fecha", F.to_date("created_at"))).cache()
valid_count, rejected_count = silver.count(), quarantine.count()
assert total == valid_count + rejected_count
assert silver.select("dispositivo", "created_at").distinct().count() == valid_count
print({"entrada": total, "silver": valid_count, "cuarentena": rejected_count, "duplicados_admisibles": duplicates.count()})
quarantine.select(F.explode("motivos").alias("motivo")).groupBy("motivo").count().show(truncate=False)


{'entrada': 15690, 'silver': 12853, 'cuarentena': 2837, 'duplicados_admisibles': 0}
+----------------+-----+
|motivo          |count|
+----------------+-----+
|fallo_sensor_gas|2837 |
+----------------+-----+



## Pruebas de las reglas
Casos sintéticos separados del dataset: valor vacío, dato no numérico, NaN, infinito, fallo de gas y extremos de humedad. Estas filas nunca se mezclan con las mediciones reales.

In [7]:
base = raw.first().asDict()
cases = [("valida", {}, True), ("vacio", {"humedad_pct":" "}, False),
         ("texto", {"humedad_pct":"abc"}, False), ("nan", {"humedad_pct":"NaN"}, False),
         ("infinito", {"gas_ohms":"Infinity"}, False), ("gas_cero", {"gas_ohms":"0"}, False),
         ("humedad_alta", {"humedad_pct":"101"}, False),
         ("fecha_mala", {"created_at":"fecha-invalida"}, False),
         ("humedad_100", {"humedad_pct":"100", "gas_ohms":"1000"}, True)]
rows = [{**base, **changes, "id": label} for label, changes, _ in cases]
results = {r.id: len(r.motivos) == 0 for r in validar(spark.createDataFrame(rows, schema)).select("id", "motivos").collect()}
assert results == {label: expected for label, _, expected in cases}, results
print("Pruebas de calidad superadas:", len(cases))


Pruebas de calidad superadas: 9


In [8]:
minute = (silver.groupBy("dispositivo", F.window("created_at", "1 minute").alias("ventana"))
    .agg(F.avg("humedad_pct").alias("humedad_pct"),
         F.avg("temperatura_c").alias("temperatura_c"),
         F.avg("presion_hpa").alias("presion_hpa"), F.count("*").alias("lecturas"))
    .withColumn("minuto", F.col("ventana.start")).drop("ventana"))
w = Window.partitionBy("dispositivo").orderBy("minuto")
gold = (minute.withColumn("anterior", F.lag("minuto").over(w))
    .withColumn("salto_segundos", F.col("minuto").cast("long") - F.col("anterior").cast("long"))
    .withColumn("nuevo_tramo", F.when(F.col("anterior").isNull() | (F.col("salto_segundos") > 600), 1).otherwise(0))
    .withColumn("tramo", F.sum("nuevo_tramo").over(w.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn("fecha", F.to_date("minuto")).withColumn("hora_lima", F.hour("minuto"))
    .drop("anterior", "nuevo_tramo")).cache()
assert gold.agg(F.sum("lecturas")).first()[0] == valid_count
silver.write.mode("errorifexists").partitionBy("fecha").parquet(str(out / "silver"))
gold.write.mode("errorifexists").partitionBy("fecha").parquet(str(out / "gold"))
quarantine.write.mode("errorifexists").parquet(str(out / "cuarentena"))
gold.orderBy("minuto").show(10, truncate=False)


+------------+-----------+-------------+-----------+--------+-------------------+--------------+-----+----------+---------+
|dispositivo |humedad_pct|temperatura_c|presion_hpa|lecturas|minuto             |salto_segundos|tramo|fecha     |hora_lima|
+------------+-----------+-------------+-----------+--------+-------------------+--------------+-----+----------+---------+
|ESP32_AireIQ|37.3878    |18.7746      |647.35     |1       |2026-10-21 23:52:00|NULL          |1    |2026-10-21|23       |
|ESP32_AireIQ|37.3368    |18.7791      |647.34     |1       |2026-10-21 23:53:00|60            |1    |2026-10-21|23       |
|ESP32_AireIQ|37.251     |18.8374      |647.33     |1       |2026-10-21 23:54:00|60            |1    |2026-10-21|23       |
|ESP32_AireIQ|37.2025    |18.8675      |647.33     |1       |2026-10-21 23:55:00|60            |1    |2026-10-21|23       |
|ESP32_AireIQ|37.1502    |18.9174      |647.32     |1       |2026-10-21 23:56:00|60            |1    |2026-10-21|23       |
|ESP32_A

In [9]:
read_silver = spark.read.parquet(str(out / "silver"))
read_gold = spark.read.parquet(str(out / "gold"))
assert read_silver.count() == valid_count
assert read_gold.count() == gold.count()
# Comparación multiconjunto: verifica valores, no solo los conteos.
for expected, actual in [(silver, read_silver), (gold, read_gold)]:
    a, b = expected.select(sorted(expected.columns)), actual.select(sorted(expected.columns))
    assert a.exceptAll(b).limit(1).count() == 0
    assert b.exceptAll(a).limit(1).count() == 0
date_value = read_silver.agg(F.min("fecha")).first()[0]
filtered = read_silver.filter(F.col("fecha") == F.lit(date_value))
filtered.explain(True)
plan = filtered._jdf.queryExecution().executedPlan().toString()
assert "PartitionFilters:" in plan and "PartitionFilters: []" not in plan
assert filtered.count() == silver.filter(F.col("fecha") == F.lit(date_value)).count()
(out / "plan_partition_filters.txt").write_text(plan, encoding="utf-8")
report = {"run_id": run, "source_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
          "spark_version": spark.version, "master": spark.sparkContext.master,
          "timezone": "America/Lima", "entrada": total, "silver": valid_count,
          "cuarentena": rejected_count, "gold_minutos": gold.count(),
          "tramos": gold.select("dispositivo", "tramo").distinct().count(),
          "pruebas_sinteticas": len(cases), "roundtrip_valores": True,
          "partition_filters": True,
          "motivos": {r.motivo:r['count'] for r in quarantine.select(F.explode("motivos").alias("motivo")).groupBy("motivo").count().collect()}}
(out / "manifest.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Evidencias:", out)
raw.unpersist(); checked.unpersist(); silver.unpersist(); quarantine.unpersist(); gold.unpersist()


== Parsed Logical Plan ==
'Filter '`=`('fecha, 2026-10-21)
+- Relation [id#6002,created_at#6003,dispositivo#6004,temperatura_c#6005,humedad_pct#6006,presion_hpa#6007,gas_ohms#6008,altura_m#6009,iaq_estimado#6010,eco2_estimado_ppm#6011,voc_indice_estimado#6012,calidad_aire#6013,fecha#6014] parquet

== Analyzed Logical Plan ==
id: string, created_at: timestamp, dispositivo: string, temperatura_c: double, humedad_pct: double, presion_hpa: double, gas_ohms: double, altura_m: double, iaq_estimado: double, eco2_estimado_ppm: double, voc_indice_estimado: double, calidad_aire: string, fecha: date
Filter (fecha#6014 = 2026-10-21)
+- Relation [id#6002,created_at#6003,dispositivo#6004,temperatura_c#6005,humedad_pct#6006,presion_hpa#6007,gas_ohms#6008,altura_m#6009,iaq_estimado#6010,eco2_estimado_ppm#6011,voc_indice_estimado#6012,calidad_aire#6013,fecha#6014] parquet

== Optimized Logical Plan ==
Filter (isnotnull(fecha#6014) AND (fecha#6014 = 2026-10-21))
+- Relation [id#6002,created_at#6003,disp

DataFrame[dispositivo: string, humedad_pct: double, temperatura_c: double, presion_hpa: double, lecturas: bigint, minuto: timestamp, salto_segundos: bigint, tramo: bigint, fecha: date, hora_lima: int]

## Lectura del resultado y límites
Silver conserva lecturas válidas; Gold tiene una fila por minuto observado. Los promedios por minuto no representan intervalos sin observaciones. La cuarentena conserva razones y contenido original para auditoría. Las frecuencias de motivos pueden solaparse.

S3 no entrena modelos ni fija todavía p5/p95. S4 calculará referencias y transformaciones solo con el periodo de entrenamiento para evitar fuga de información. Este entorno ejecuta Spark en dos hilos locales; no demuestra un despliegue multinodo.


In [10]:
# Enlazar S4 con la salida de ESTA ejecución S3, aunque existan otras corridas.
s03_integrado = out
# Evidencia explícita de nulos y duplicados antes/después.
essential = ["id", "created_at", "dispositivo", "temperatura_c", "humedad_pct", "presion_hpa", "gas_ohms"]
print("Nulos o vacíos de entrada:")
raw.select(*[F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c) for c in essential]).show()
print("Nulos después:")
read_silver.select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in essential]).show()
print("Duplicados antes:", raw.count() - raw.select("dispositivo", "created_at").distinct().count())
print("Duplicados después:", read_silver.count() - read_silver.select("dispositivo", "created_at").distinct().count())
# Prueba independiente de la prioridad de deduplicación con dos eventos repetidos.
first = raw.first().asDict()
test_dups = spark.createDataFrame([{**first, "id":"200"}, {**first, "id":"100"}], schema)
rank_test = validar(test_dups).filter(F.size("motivos") == 0).withColumn("orden", F.row_number().over(order))
assert rank_test.filter("orden = 1").select("id").first()[0] == "100"
assert rank_test.filter("orden > 1").count() == 1
print("Deduplicación determinista sintética: OK")


Nulos o vacíos de entrada:
+---+----------+-----------+-------------+-----------+-----------+--------+
| id|created_at|dispositivo|temperatura_c|humedad_pct|presion_hpa|gas_ohms|
+---+----------+-----------+-------------+-----------+-----------+--------+
|  0|         0|          0|            0|          0|          0|       0|
+---+----------+-----------+-------------+-----------+-----------+--------+

Nulos después:


+---+----------+-----------+-------------+-----------+-----------+--------+
| id|created_at|dispositivo|temperatura_c|humedad_pct|presion_hpa|gas_ohms|
+---+----------+-----------+-------------+-----------+-----------+--------+
|  0|         0|          0|            0|          0|          0|       0|
+---+----------+-----------+-------------+-----------+-----------+--------+

Duplicados antes: 0


Duplicados después: 0
Deduplicación determinista sintética: OK


# AireIQ · S4: regresión de humedad con MLlib
**Responsable: Eberth Emmanuel Rosales Castillo.**

Pregunta U1: ¿Qué tan precisamente se estima la humedad relativa usando temperatura, presión y hora del mismo instante?

Se utiliza Gold de S3, con una observación por minuto. Esta estimación contemporánea no es el pronóstico a un minuto de U2 y no demuestra causalidad.

Método: primer 60 % cronológico para entrenar, siguiente 20 % para seleccionar por RMSE y último 20 % para prueba final. Se preservan timestamps completos en cada corte. La separación temporal es una decisión propia frente al ejemplo aleatorio de U1: reduce la mezcla de mediciones vecinas entre entrenamiento y prueba. La prueba no elige hiperparámetros. Se vuelve a ajustar la configuración ganadora con el primer 80 % y se evalúa una vez en el último 20 %.

Se comparan LinearRegression, Ridge, Elastic Net y RandomForestRegressor, más una referencia constante (media de entrenamiento). RMSE y MAE se expresan en puntos porcentuales de humedad; R² puede ser negativo. No se presupone que los modelos superen la referencia.

Referencia: https://262bigdata.github.io/lambda26/proyecto-sello/u1-producto/


In [11]:
from pathlib import Path
from datetime import datetime, timezone
import json, math, hashlib
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (SparkSession.builder.appName("AireIQ-S4-Humedad").master("local[2]")
    .config("spark.sql.session.timeZone", "America/Lima")
    .config("spark.sql.shuffle.partitions", "4").getOrCreate())
# Elegir solo ejecuciones S3 terminadas, con manifiesto y Parquet.
root = Path("/opt/aireiq/outputs")
sources = sorted(p for p in (root / "s03").iterdir() if (p / "manifest.json").is_file() and (p / "gold/_SUCCESS").is_file())
assert sources, "Ejecutar S3 primero"
source = s03_integrado
assert source in sources, "La corrida S3 integrada no esta completa"
s03 = json.loads((source / "manifest.json").read_text())
assert s03["roundtrip_valores"] and s03["partition_filters"]
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = root / "s04" / run_id
out.mkdir(parents=True, exist_ok=False)
data = spark.read.parquet(str(source / "gold")).cache()
times = [r.minuto for r in data.select("minuto").distinct().orderBy("minuto").collect()]
assert len(times) >= 100, "Historial insuficiente para tres bloques"
cut1, cut2 = times[int(len(times)*.6)], times[int(len(times)*.8)]
train = data.filter(F.col("minuto") < F.lit(cut1)).cache()
validation = data.filter((F.col("minuto") >= F.lit(cut1)) & (F.col("minuto") < F.lit(cut2))).cache()
test = data.filter(F.col("minuto") >= F.lit(cut2)).cache()
counts = {"train": train.count(), "validation": validation.count(), "test": test.count()}
assert min(counts.values()) > 0 and sum(counts.values()) == data.count()
for left, right in [(train, validation), (validation, test)]:
    assert left.agg(F.max("minuto")).first()[0] < right.agg(F.min("minuto")).first()[0]
print("S3:", source.name, "Filas por bloque:", counts)
# Mostrar cortes como strings en zona de Lima; internamente los timestamps son instantes UTC.
for label, frame in [("train", train), ("validation", validation), ("test", test)]:
    print(label, frame.select(F.min("minuto").cast("string"), F.max("minuto").cast("string")).first())


26/11/20 03:09:41 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


S3: 20261120T030852967375Z Filas por bloque: {'train': 2535, 'validation': 845, 'test': 845}
train Row(CAST(min(minuto) AS STRING)='2026-10-21 23:52:00', CAST(max(minuto) AS STRING)='2026-11-06 02:57:00')
validation Row(CAST(min(minuto) AS STRING)='2026-11-06 02:58:00', CAST(max(minuto) AS STRING)='2026-11-18 06:16:00')
test Row(CAST(min(minuto) AS STRING)='2026-11-18 06:17:00', CAST(max(minuto) AS STRING)='2026-11-19 20:50:00')


## Preparación y comparación
La hora se representa con seno y coseno, de modo que 23:59 y 00:00 sean cercanas. Estas transformaciones y VectorAssembler forman parte del modelo guardado. No se usan humedad actual ni índices derivados como predictores del objetivo contemporáneo.


In [12]:
features = ["temperatura_c", "presion_hpa", "hora_sin", "hora_cos"]
sql = "SELECT *, sin(2 * PI() * (hour(minuto) + minute(minuto)/60.0) / 24) AS hora_sin, cos(2 * PI() * (hour(minuto) + minute(minuto)/60.0) / 24) AS hora_cos FROM __THIS__"
def pipeline(name):
    regressors = {
        "lineal": LinearRegression(labelCol="humedad_pct", regParam=0.0, maxIter=100),
        "ridge": LinearRegression(labelCol="humedad_pct", regParam=0.1, elasticNetParam=0.0, maxIter=100),
        "elastic_net": LinearRegression(labelCol="humedad_pct", regParam=0.1, elasticNetParam=0.5, maxIter=100),
        "random_forest": RandomForestRegressor(labelCol="humedad_pct", numTrees=50, maxDepth=6, seed=42)
    }
    return Pipeline(stages=[SQLTransformer(statement=sql), VectorAssembler(inputCols=features, outputCol="features", handleInvalid="error"), regressors[name]])
def metrics(pred):
    result = {m: RegressionEvaluator(labelCol="humedad_pct", predictionCol="prediction", metricName=m).evaluate(pred) for m in ["rmse", "mae", "r2"]}
    assert all(math.isfinite(v) for v in result.values()), result
    return result

mean_train = train.agg(F.avg("humedad_pct")).first()[0]
baseline_val = metrics(validation.withColumn("prediction", F.lit(mean_train)))
results = []
for name in ["lineal", "ridge", "elastic_net", "random_forest"]:
    fitted = pipeline(name).fit(train)
    pred = fitted.transform(validation).cache()
    score = metrics(pred)
    results.append({"modelo": name, **score})
    pred.unpersist()
    print(name, score, flush=True)
winner = min(results, key=lambda r: (r["rmse"], r["modelo"]))["modelo"]
spark.createDataFrame(results + [{"modelo":"referencia_media", **baseline_val}]).select("modelo", "rmse", "mae", "r2").orderBy("rmse").show(truncate=False)
print("Mejor configuración ML en validación:", winner)


26/11/20 03:09:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/20 03:09:46 WARN Instrumentation: [3b8478bf] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


lineal {'rmse': 5.667646023891026, 'mae': 4.648312388556594, 'r2': 0.35746933657961477}
ridge {'rmse': 5.71078635955574, 'mae': 4.616513301405425, 'r2': 0.34765062790210133}
elastic_net {'rmse': 5.736864846366921, 'mae': 4.586194152304845, 'r2': 0.34167907617839455}
random_forest {'rmse': 4.329596766105983, 'mae': 2.9952414602934816, 'r2': 0.6250413967543406}
+----------------+-----------------+------------------+--------------------+
|modelo          |rmse             |mae               |r2                  |
+----------------+-----------------+------------------+--------------------+
|random_forest   |4.329596766105983|2.9952414602934816|0.6250413967543406  |
|lineal          |5.667646023891026|4.648312388556594 |0.35746933657961477 |
|ridge           |5.71078635955574 |4.616513301405425 |0.34765062790210133 |
|elastic_net     |5.736864846366921|4.586194152304845 |0.34167907617839455 |
|referencia_media|7.354722419318235|5.277439542714421 |-0.08198438778585349|
+----------------+----

In [13]:
development = train.unionByName(validation).cache()
final_model = pipeline(winner).fit(development)
pred_test = final_model.transform(test).cache()
test_metrics = metrics(pred_test)
mean_dev = development.agg(F.avg("humedad_pct")).first()[0]
baseline_test = metrics(test.withColumn("prediction", F.lit(mean_dev)))
final_model.write().save(str(out / "modelo_humedad_u1"))
loaded = PipelineModel.load(str(out / "modelo_humedad_u1"))
reloaded = loaded.transform(test).select("dispositivo", "minuto", F.col("prediction").alias("restored"))
delta = pred_test.select("dispositivo", "minuto", "prediction").join(reloaded, ["dispositivo", "minuto"]).select(F.max(F.abs(F.col("prediction")-F.col("restored")))).first()[0]
assert delta is not None and delta <= 1e-9, delta
assert pred_test.count() == test.count()
assert pred_test.filter(F.col("prediction").isNull() | F.isnan("prediction")).count() == 0
pred_test.select("dispositivo", "minuto", "fecha", "humedad_pct", "prediction").write.partitionBy("fecha").parquet(str(out / "predicciones_test"))
print("Prueba final:", test_metrics)
print("Referencia media:", baseline_test)
print("Supera referencia en RMSE:", test_metrics["rmse"] < baseline_test["rmse"])
print("Máxima diferencia tras recargar:", delta)


Prueba final: {'rmse': 4.098491478587124, 'mae': 3.2174829868051806, 'r2': -2.6561100188092888}
Referencia media: {'rmse': 3.211229321899784, 'mae': 2.4930445803901726, 'r2': -1.244470101142479}
Supera referencia en RMSE: False
Máxima diferencia tras recargar: 0.0


## Indicador descriptivo
Referencia estadística: percentiles p5/p95 de los minutos del primer 60 % del historial, calculados solo con entrenamiento. Este periodo es una decisión metodológica explícita, pendiente de aprobación académica; no se reutilizan los umbrales de IAQ del brief. El indicador solicitado se calcula sobre **lecturas válidas Silver**, con denominador por día/hora. Se entrega además una vista por minuto, que reduce el sesgo de frecuencia. Un porcentaje de lecturas no equivale a un porcentaje de tiempo.


In [14]:
p5, p95 = train.approxQuantile("humedad_pct", [.05, .95], 0.0)
def describe(frame, timestamp, unit, keys):
    tagged = (frame.withColumn("fecha", F.to_date(timestamp)).withColumn("hora_lima", F.hour(timestamp))
        .withColumn("fuera", ((F.col("humedad_pct") < p5) | (F.col("humedad_pct") > p95)).cast("int")))
    return (tagged.groupBy(*keys).agg(F.count("*").alias(unit), F.avg("humedad_pct").alias("humedad_media_pct"),
        (100*F.avg("fuera")).alias("porcentaje_fuera_referencia")))
silver = spark.read.parquet(str(source / "silver"))
daily = describe(silver, "created_at", "lecturas_validas", ["fecha"])
hourly = describe(silver, "created_at", "lecturas_validas", ["hora_lima"])
minute_summary = describe(data, "minuto", "minutos_observados", ["fecha"])
daily.write.parquet(str(out / "descriptivo_diario_lecturas"))
hourly.write.parquet(str(out / "descriptivo_horario_lecturas"))
minute_summary.write.parquet(str(out / "descriptivo_diario_minutos"))
print("Referencia humedad (%):", {"p5":p5, "p95":p95})
daily.orderBy("fecha").show(40, truncate=False)
hourly.orderBy("hora_lima").show(24, truncate=False)
report = {"run_id": run_id, "s03_run": source.name, "source_sha256": s03["source_sha256"],
    "spark_version": spark.version, "timezone": "America/Lima", "objetivo": "humedad contemporanea media por minuto",
    "split": "cronologico 60/20/20", "conteos": counts, "features": features,
    "validacion": results, "baseline_validacion": baseline_val, "modelo_elegido": winner,
    "test": test_metrics, "baseline_test": baseline_test,
    "supera_baseline_test": test_metrics["rmse"] < baseline_test["rmse"],
    "diferencia_recarga": delta, "p5": p5, "p95": p95,
    "referencia": "primer 60% de minutos observados; no umbral sanitario",
    "fuera_rango_predicciones_test": pred_test.filter(~F.col("prediction").between(0,100)).count()}
(out / "manifest.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Evidencias:", out)
for frame in [data, train, validation, test, development, pred_test]: frame.unpersist()


Referencia humedad (%): {'p5': 30.283099999999997, 'p95': 42.969674999999995}


+----------+----------------+------------------+---------------------------+
|fecha     |lecturas_validas|humedad_media_pct |porcentaje_fuera_referencia|
+----------+----------------+------------------+---------------------------+
|2026-10-21|5               |37.265660000000004|0.0                        |
|2026-10-22|771             |39.45005343709464 |0.38910505836575876        |
|2026-10-23|10              |39.5558           |0.0                        |
|2026-10-28|195             |35.760983589743596|0.0                        |
|2026-10-29|2255            |34.54182700665187 |4.878048780487805          |
|2026-10-30|687             |40.091847889374094|66.95778748180496          |
|2026-10-31|145             |31.591694482758626|27.586206896551722         |
|2026-11-01|1411            |32.64578596739898 |11.76470588235294          |
|2026-11-05|378             |36.44720502645502 |0.0                        |
|2026-11-06|1825            |42.698310575342504|55.83561643835616          |

[Stage 408:>                                                        (0 + 2) / 2]

+---------+----------------+------------------+---------------------------+
|hora_lima|lecturas_validas|humedad_media_pct |porcentaje_fuera_referencia|
+---------+----------------+------------------+---------------------------+
|0        |584             |36.875035616438375|0.0                        |
|1        |535             |38.27341401869165 |0.0                        |
|2        |529             |39.57659376181475 |0.0                        |
|3        |592             |39.43710016891897 |37.16216216216216          |
|4        |397             |40.46016120906802 |26.700251889168765         |
|5        |455             |41.24385230769232 |35.38461538461539          |
|6        |367             |37.736055858310614|7.084468664850137          |
|7        |291             |34.690074226804136|0.3436426116838488         |
|8        |389             |35.17879691516709 |19.28020565552699          |
|9        |716             |42.6144675977654  |68.29608938547486          |
|10       |8

## Interpretación para sustentación
- RMSE penaliza más los errores grandes; MAE expresa el error absoluto promedio.
- R² negativo indica peor ajuste que predecir la media del conjunto evaluado. La referencia operativa usa la media del entrenamiento, no la del test.
- El modelo elegido es el mejor ML de la comparación en validación; su utilidad depende de la prueba final y de la comparación con la referencia. Guardarlo no demuestra que sea útil en producción.
- Los huecos y el único sensor limitan la generalización. No se modifican métricas ni se cambia el ganador mirando el test.
- U2 requerirá otro objetivo: humedad de t+1 minuto, pares temporales correctos y comparación con persistencia. Este cuaderno no implementa esa inferencia.


In [15]:
# Manifiesto final de S5: solo se escribe después de superar todas las celdas.
import getpass
from IPython.display import display, Markdown
final_stage = final_model.stages[-1]
if hasattr(final_stage, "featureImportances"):
    importance = dict(zip(features, map(float, final_stage.featureImportances)))
    print("Importancias del modelo, no efectos causales:", importance)
else:
    importance = None
end_report = {"s03": str(s03_integrado), "s04": str(out), "completed_utc": datetime.now(timezone.utc).isoformat(),
              "executed_by_container_user": getpass.getuser(), "responsable_academico": "Eberth Emmanuel Rosales Castillo",
              "source_sha256": s03["source_sha256"], "modelo": winner, "test": test_metrics,
              "baseline_test": baseline_test, "importancias": importance, "duplicate_fixture_passed": True,
              "status": "ejecucion_tecnica_completa; sustentacion y capturas pendientes"}
(out / "s05_manifest.json").write_text(json.dumps(end_report, ensure_ascii=False, indent=2), encoding="utf-8")
display(Markdown("**Ejecución técnica terminada.** Revisar las métricas y sus limitaciones antes de sustentar."))
print(json.dumps(end_report, ensure_ascii=False, indent=2))


Importancias del modelo, no efectos causales: {'temperatura_c': 0.32538692427636334, 'presion_hpa': 0.2435432115289446, 'hora_sin': 0.33977539069457213, 'hora_cos': 0.09129447350011997}


**Ejecución técnica terminada.** Revisar las métricas y sus limitaciones antes de sustentar.

{
  "s03": "/opt/aireiq/outputs/s03/20261120T030852967375Z",
  "s04": "/opt/aireiq/outputs/s04/20261120T030941557986Z",
  "completed_utc": "2026-11-20T03:10:22.329587+00:00",
  "executed_by_container_user": "root",
  "responsable_academico": "Eberth Emmanuel Rosales Castillo",
  "source_sha256": "35fe207bedce69285c7eae033d7bdc4bccc3dfc6bf4055c231b7a8a2eb4583c4",
  "modelo": "random_forest",
  "test": {
    "rmse": 4.098491478587124,
    "mae": 3.2174829868051806,
    "r2": -2.6561100188092888
  },
  "baseline_test": {
    "rmse": 3.211229321899784,
    "mae": 2.4930445803901726,
    "r2": -1.244470101142479
  },
  "importancias": {
    "temperatura_c": 0.32538692427636334,
    "presion_hpa": 0.2435432115289446,
    "hora_sin": 0.33977539069457213,
    "hora_cos": 0.09129447350011997
  },
  "duplicate_fixture_passed": true,
  "status": "ejecucion_tecnica_completa; sustentacion y capturas pendientes"
}
